In [20]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [21]:
escalada = pd.read_csv("../docs/data/matriz_partidos_scaled_pca.csv")
partidos = pd.read_csv("../data/partidos_primera_ronda.csv")

In [22]:
escalada = pd.merge(escalada,partidos.rename(columns={'id':'id_partido'})[['id_partido','date','time_utc']], on='id_partido', how='left')
escalada['date'] = pd.to_datetime(escalada['date']).dt.date
escalada['time_utc'] = pd.to_datetime(escalada['time_utc'], format='%H:%M:%S').dt.time
escalada.to_csv("../docs/data/matriz_partidos_scaled_pca.csv", index=False)

In [ ]:
escalada['date'] = pd.to_datetime(escalada['date']).dt.date
escalada['time_utc'] = pd.to_datetime(escalada['time_utc'], format='%H:%M:%S').dt.time


,time_utc
12,22:00:00
9,22:00:00
50,22:00:00
31,23:00:00
27,23:00:00
66,23:00:00
69,23:00:00
35,23:00:00
64,23:30:00
65,23:30:00


In [13]:
partidos['partido'] = partidos['home_team'] + " vs " + partidos['away_team']
partidos[['partido','time_utc']]

,partido,time_utc
0,Mexico vs South Africa,19:00:00
1,Korea Republic vs Czechia,02:00:00
2,Canada vs Bosnia-Herzegovina,19:00:00
3,USA vs Paraguay,01:00:00
4,Qatar vs Switzerland,19:00:00
...,...,...
67,Croatia vs Ghana,21:00:00
68,Congo DR vs Uzbekistan,23:30:00
69,Colombia vs Portugal,23:30:00
70,Algeria vs Austria,02:00:00


In [ ]:
escalada['partido'] = escalada['home_team'] + " vs " + escalada['away_team']


['Korea Republic vs Czechia',
 'Mexico vs South Africa',
 'Czechia vs South Africa',
 'Mexico vs Korea Republic',
 'South Africa vs Korea Republic',
 'Czechia vs Mexico',
 'Canada vs Bosnia-Herzegovina',
 'Switzerland vs Bosnia-Herzegovina',
 'Qatar vs Switzerland',
 'Canada vs Qatar',
 'Bosnia-Herzegovina vs Qatar',
 'Switzerland vs Canada',
 'Brazil vs Morocco',
 'Scotland vs Morocco',
 'Haiti vs Scotland',
 'Brazil vs Haiti',
 'Morocco vs Haiti',
 'Scotland vs Brazil',
 'Australia vs Turkey',
 'USA vs Paraguay',
 'Turkey vs Paraguay',
 'USA vs Australia',
 'Paraguay vs Australia',
 'Turkey vs USA',
 'Ecuador vs Curaçao',
 'Germany vs Curaçao',
 'Ecuador vs Germany',
 "Côte d'Ivoire vs Ecuador",
 "Germany vs Côte d'Ivoire",
 "Curaçao vs Côte d'Ivoire",
 'Netherlands vs Sweden',
 'Japan vs Sweden',
 'Sweden vs Tunisia',
 'Netherlands vs Japan',
 'Tunisia vs Japan',
 'Tunisia vs Netherlands',
 'IR Iran vs New Zealand',
 'Belgium vs IR Iran',
 'Egypt vs IR Iran',
 'Belgium vs Egypt',
 '

In [3]:
elos.head()

,Rank,Team,ELO_Rating,Avg_Rating,1_Year_Change,Total_Matches,Home,Away,Neutral,Wins,Losses,Draws,Goals_For,Goals_Against
0,1,Spain,2165,1946,+15,780,340,302,138,461,138,181,1591,697
1,2,Argentina,2113,1987,-28,1109,381,419,309,610,228,271,2112,1136
2,3,France,2081,1795,+51,938,473,341,124,474,269,195,1706,1272
3,4,England,2020,1983,+9,1160,509,524,127,683,215,262,2719,1118
4,5,Brazil,1984,1998,-9,1065,372,346,347,670,172,223,2294,954


In [4]:
mapping = {
    'South Korea': 'Korea Republic',
    'United States': 'USA',
    'Ivory Coast': 'Côte d\'Ivoire',
    'Iran': 'IR Iran',
    'Bosnia and Herzegovina':'Bosnia-Herzegovina',
    'Cape Verde':'Cabo Verde',
    'Congo':'Congo DR'   
}

In [5]:
elos['Team'] = elos['Team'].replace(mapping)

In [6]:
teams_elo = elos['Team'].unique().tolist()

In [7]:
teams_partidos = pd.Series(partidos['home_team'].tolist() +  partidos['away_team'].tolist()).drop_duplicates().to_list()

In [8]:
teams_partidos_elo = []
for e in teams_partidos:
    if e in teams_elo:
        teams_partidos_elo.append(e)

In [9]:
elos = elos[elos['Team'].isin(teams_partidos_elo)]

In [10]:
df = (
    partidos
    .merge(
        elos[['Team','ELO_Rating']].rename(columns={'Team':'home_team','ELO_Rating':'home_elo_rating'}), 
        on='home_team', 
        how='left')
    .merge(
        elos[['Team','ELO_Rating']].rename(columns={'Team':'away_team','ELO_Rating':'away_elo_rating'}), 
        on='away_team', 
        how='left')
)

In [11]:
df['match_rating'] = df['home_elo_rating'] + df['away_elo_rating']
df['elo_diff'] = abs(df['home_elo_rating'] - df['away_elo_rating'])
df.sort_values(by='match_rating', ascending=False).head(10)

,id,match_number,group_name,home_team_id,home_team,home_team_code,away_team_id,away_team,away_team_code,stadium_id,stadium,stadium_city,stadium_country,date,time_utc,home_elo_rating,away_elo_rating,match_rating,elo_diff
63,46,66,H,31,Uruguay,URU,30,Spain,ESP,16,Estadio Akron,Guadalajara,Mexico,2026-06-27,00:00:00,1892,2165,4057,273
61,54,61,I,35,Norway,NOR,33,France,FRA,9,Gillette Stadium,"Foxborough, MA",USA,2026-06-26,19:00:00,1912,2081,3993,169
69,66,71,K,43,Colombia,COL,41,Portugal,POR,5,Hard Rock Stadium,"Miami Gardens, FL",USA,2026-06-27,23:30:00,1975,1984,3959,9
16,52,17,I,33,France,FRA,34,Senegal,SEN,1,MetLife Stadium,"East Rutherford, NJ",USA,2026-06-16,19:00:00,2081,1878,3959,203
21,68,22,L,47,England,ENG,48,Croatia,CRO,2,AT&T Stadium,"Arlington, TX",USA,2026-06-17,20:00:00,2020,1930,3950,90
40,56,43,J,37,Argentina,ARG,39,Austria,AUT,2,AT&T Stadium,"Arlington, TX",USA,2026-06-22,17:00:00,2113,1827,3940,286
9,34,11,F,21,Netherlands,NED,22,Japan,JPN,2,AT&T Stadium,"Arlington, TX",USA,2026-06-14,20:00:00,1961,1904,3865,57
18,58,19,J,37,Argentina,ARG,38,Algeria,ALG,8,Arrowhead Stadium,"Kansas City, MO",USA,2026-06-17,01:00:00,2113,1743,3856,370
55,27,56,E,18,Ecuador,ECU,19,Germany,GER,1,MetLife Stadium,"East Rutherford, NJ",USA,2026-06-25,20:00:00,1933,1923,3856,10
5,13,7,C,9,Brazil,BRA,12,Morocco,MAR,1,MetLife Stadium,"East Rutherford, NJ",USA,2026-06-13,22:00:00,1984,1821,3805,163


In [92]:
df.to_csv('../data/partidos_rankeados.csv', index=False)